# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
# Установка библиотек, если ещё не установлены
%pip install -q requests beautifulsoup4 schedule pytest

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import urljoin

import requests
import schedule
from bs4 import BeautifulSoup

BASE_URL = 'http://books.toscrape.com/'
CATALOGUE_PAGE_URL = BASE_URL + 'catalogue/page-{}.html'
RATING_WORDS = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}


def get_soup(url: str, session: requests.Session = None) -> BeautifulSoup:
    """
    Загружает страницу и возвращает её разобранный HTML.

    Args:
        url (str): адрес страницы.
        session (requests.Session | None): HTTP-сессия для повторного
            использования соединений; если None, используется requests.get.

    Returns:
        BeautifulSoup: дерево HTML-документа.

    Raises:
        requests.HTTPError: если сервер вернул код ошибки (например, 404).
    """
    getter = session.get if session is not None else requests.get
    response = getter(url, timeout=15)
    response.raise_for_status()
    response.encoding = 'utf-8'  # сайт не указывает кодировку в заголовках
    return BeautifulSoup(response.text, 'html.parser')

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [ ]:
def get_book_data(book_url: str, session: requests.Session = None) -> dict:
    """
    Собирает данные об одной книге с её страницы на books.toscrape.com.

    Извлекает название, цену, рейтинг, количество в наличии, описание,
    категорию и все характеристики из таблицы Product Information.

    Args:
        book_url (str): полный адрес страницы книги.
        session (requests.Session | None): необязательная HTTP-сессия.

    Returns:
        dict: словарь с ключами
            'title' (str) — название;
            'price' (float) — цена в фунтах;
            'rating' (int) — рейтинг от 1 до 5 (0, если не найден);
            'available' (int) — количество экземпляров в наличии;
            'description' (str) — описание (пустая строка, если нет);
            'category' (str) — категория;
            'url' (str) — адрес страницы книги;
            'product_info' (dict) — пары «характеристика: значение»
            из таблицы Product Information.
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    soup = get_soup(book_url, session)
    main = soup.find('div', class_='product_main')

    title = main.h1.get_text(strip=True)
    price_text = main.find('p', class_='price_color').get_text()
    price = float(re.sub(r'[^\d.]', '', price_text))

    rating = 0
    rating_tag = main.find('p', class_='star-rating')
    if rating_tag is not None:
        for css_class in rating_tag.get('class', []):
            rating = RATING_WORDS.get(css_class, rating)

    description = ''
    description_header = soup.find('div', id='product_description')
    if description_header is not None:
        description = description_header.find_next_sibling('p').get_text(
            strip=True)

    breadcrumb = soup.select('ul.breadcrumb li a')
    category = breadcrumb[-1].get_text(strip=True) if breadcrumb else ''

    product_info = {
        row.th.get_text(strip=True): row.td.get_text(strip=True)
        for row in soup.select('table.table-striped tr')
    }
    available = re.search(r'(\d+)', product_info.get('Availability', '0'))

    return {
        'title': title,
        'price': price,
        'rating': rating,
        'available': int(available.group(1)) if available else 0,
        'description': description,
        'category': category,
        'url': book_url,
        'product_info': product_info,
    }
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [4]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': 51.77,
 'rating': 3,
 'available': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up th

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [ ]:
def scrape_books(is_save: bool = False, max_pages: int = None,
                 output_path: str = 'books_data.txt',
                 max_workers: int = 10) -> list:
    """
    Собирает данные обо всех книгах каталога books.toscrape.com.

    Проходит в цикле по страницам вида
    http://books.toscrape.com/catalogue/page-{N}.html, начиная с первой,
    пока сайт не вернёт 404 (или не достигнут max_pages), и для каждой
    книги вызывает get_book_data. Страницы книг с одной страницы каталога
    загружаются параллельно в несколько потоков.

    Args:
        is_save (bool): если True, результат сохраняется в текстовый файл
            output_path (одна книга — одна строка JSON); иначе не сохраняется.
        max_pages (int | None): ограничение числа страниц каталога;
            None — обойти все страницы.
        output_path (str): путь к файлу (по умолчанию books_data.txt
            в текущей папке).
        max_workers (int): число потоков для загрузки страниц книг.

    Returns:
        list[dict]: список словарей с данными книг (см. get_book_data).
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    books = []
    page = 1
    with requests.Session() as session, \
            ThreadPoolExecutor(max_workers=max_workers) as executor:
        while max_pages is None or page <= max_pages:
            page_url = CATALOGUE_PAGE_URL.format(page)
            try:
                soup = get_soup(page_url, session)
            except requests.HTTPError as error:
                if error.response.status_code == 404:
                    break  # страницы каталога закончились
                raise
            book_urls = [urljoin(page_url, link['href'])
                         for link in soup.select('article.product_pod h3 a')]
            books.extend(executor.map(
                lambda url: get_book_data(url, session), book_urls))
            page += 1

    if is_save:
        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8') as file:
            for book in books:
                file.write(json.dumps(book, ensure_ascii=False) + '\n')
    return books
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [6]:
# Проверка работоспособности функции
OUTPUT_PATH = '../artifacts/books_data.txt'
start = time.time()
res = scrape_books(is_save=True, output_path=OUTPUT_PATH)
print(f'Время сбора: {time.time() - start:.1f} с')
print(type(res), len(res))
print('Ключи:', list(res[0].keys()))
print('Уникальных названий по URL:', len({book['url'] for book in res}))
with open(OUTPUT_PATH, encoding='utf-8') as file:
    print('Строк в файле:', sum(1 for _ in file))
res[-1]

Время сбора: 36.7 с
<class 'list'> 1000
Ключи: ['title', 'price', 'rating', 'available', 'description', 'category', 'url', 'product_info']
Уникальных названий по URL: 1000
Строк в файле: 1000


{'title': '1,000 Places to See Before You Die',
 'price': 26.08,
 'rating': 5,
 'available': 1,
 'description': 'Around the World, continent by continent, here is the best the world has to offer: 1,000 places guaranteed to give travelers the shivers. Sacred ruins, grand hotels, wildlife preserves, hilltop villages, snack shacks, castles, festivals, reefs, restaurants, cathedrals, hidden islands, opera houses, museums, and more. Each entry tells exactly why it\'s essential to visit. Th Around the World, continent by continent, here is the best the world has to offer: 1,000 places guaranteed to give travelers the shivers. Sacred ruins, grand hotels, wildlife preserves, hilltop villages, snack shacks, castles, festivals, reefs, restaurants, cathedrals, hidden islands, opera houses, museums, and more. Each entry tells exactly why it\'s essential to visit. Then come the nuts and bolts: addresses, websites, phone and fax numbers, best times to visit. Stop dreaming and get going.This hefty vo

## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
def scheduled_job(output_path: str = OUTPUT_PATH) -> None:
    books = scrape_books(is_save=True, output_path=output_path)
    print(f'[{datetime.now():%Y-%m-%d %H:%M:%S}] собрано {len(books)} книг, '
          f'сохранено в {output_path}')


def run_scheduler(run_at: str = '19:00', check_interval: int = 60,
                  max_runs: int = None) -> None:
    """
    Запускает ежедневный сбор данных в заданное время.

    В бесконечном цикле раз в check_interval секунд проверяет расписание,
    чтобы не нагружать систему постоянными проверками.

    Args:
        run_at (str): время запуска в формате 'чч:мм'.
        check_interval (int): пауза между проверками, секунды.
        max_runs (int | None): остановиться после указанного числа запусков
            (для демонстрации в ноутбуке); None — работать бесконечно.
    """
    runs = 0

    def job() -> None:
        nonlocal runs
        scheduled_job()
        runs += 1

    schedule.clear()
    schedule.every().day.at(run_at).do(job)
    print(f'Планировщик запущен: сбор каждый день в {run_at}, '
          f'следующий запуск {schedule.next_run():%Y-%m-%d %H:%M}')
    while max_runs is None or runs < max_runs:
        schedule.run_pending()
        time.sleep(check_interval)
    schedule.clear()


# Боевой режим — каждый день в 19:00 (бесконечный цикл):
# run_scheduler('19:00')

# Локальная проверка: запуск через минуту, проверка раз в 10 секунд,
# остановка после первого выполнения
test_time = (datetime.now() + timedelta(minutes=1)).strftime('%H:%M')
run_scheduler(test_time, check_interval=10, max_runs=1)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

Планировщик запущен: сбор каждый день в 23:37, следующий запуск 2026-11-15 23:37


KeyboardInterrupt: 

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [ ]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
import sys
! "{sys.executable}" -m pytest ../tests/test_scraper.py -v --color=no

============================= test session starts =============================
platform win32 -- Python 3.13.13, pytest-9.2.0, pluggy-1.7.0 -- venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: books_scraper
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 6 items

..\tests\test_scraper.py::test_book_data_is_dict_with_keys PASSED        [ 16%]
..\tests\test_scraper.py::test_book_field_values PASSED                  [ 33%]
..\tests\test_scraper.py::test_product_info_table PASSED                 [ 50%]
..\tests\test_scraper.py::test_catalogue_page_has_20_links PASSED        [ 66%]
..\tests\test_scraper.py::test_scrape_books_limited_and_saved PASSED     [ 83%]
..\tests\test_scraper.py::test_scrape_books_without_saving PASSED        [100%]

============================== 6 passed in 4.28s ==============================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Были заданы задачи на практику запросов и парсинга страниц`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Отправка запросов и парсинг получилось реализовать сразу. Документацию + настройка шедулера - с помощью ИИ`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: При настройке шедулера и написании документации использовался ИИ`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: Всё понятно`

**5. Самооценка по критериям задания**:

`Ответ: ; из 5`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: Немного запутанный алгоритм работы с гитом`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

